# Standalone EVT

An Edge–Vertex Transformer (EVT) updates one state per vertex and one state per ordered vertex pair. This example runs a single EVT layer on a small graph on CPU.

## Run a single layer on embedded states

With batch size 1, three vertices, and width 8, `X` has shape `(1, 3, 8)` and `Y` has shape `(1, 3, 3, 8)`. The first pair index is the source vertex and the second is the target. All three vertices are valid in this example.

In [ ]:
import torch
from path_augmented_transformer import EdgeVertexTransformerLayer

# Set up embedded states for three vertices and their ordered pairs.
# The first pair channel marks directed edges.
torch.manual_seed(7)
X = torch.randn(1, 3, 8)
Y = torch.zeros(1, 3, 3, 8)
Y[0, 0, 1, 0] = 1.0  # Directed edge 0 -> 1 in channel 0
Y[0, 1, 2, 0] = 1.0  # Directed edge 1 -> 2 in channel 0
node_mask = torch.ones(1, 3, dtype=torch.bool)

# Apply one EVT layer directly to the width-8 states.
block = EdgeVertexTransformerLayer(d_model=8, num_heads=2).eval()
with torch.inference_mode():
    X_next, Y_next = block(X, Y, node_mask)

# Check that both streams keep their shapes.
assert X_next.shape == X.shape and Y_next.shape == Y.shape
print(f'Vertex states {tuple(X_next.shape)}')
print(f'Ordered-pair states {tuple(Y_next.shape)}')

## Start from raw features

`EdgeVertexTransformer` adds input projections for vertex and pair features. With `num_layers=1`, it projects each stream to width 8 and applies one EVT layer. Here each vertex has one raw feature and each ordered pair has one adjacency feature. You can run this cell independently.

In [ ]:
import torch
from path_augmented_transformer import EdgeVertexTransformer

# Create fresh inputs with one raw feature per vertex and one per pair.
X_raw = torch.tensor([[[1.0], [2.0], [3.0]]])
Y_raw = torch.zeros(1, 3, 3, 1)
Y_raw[0, 0, 1, 0] = 1.0  # Directed edge 0 -> 1
Y_raw[0, 1, 2, 0] = 1.0  # Directed edge 1 -> 2
node_mask = torch.ones(1, 3, dtype=torch.bool)

# Project both streams to width 8 and apply one EVT layer.
encoder = EdgeVertexTransformer(d_x=1, d_y=1, d_h=8, num_layers=1, num_heads=2).eval()
with torch.inference_mode():
    out = encoder(X_raw, Y_raw, node_mask)

# Check the projected width and the vertex axes of each stream.
assert out.X.shape == (1, 3, 8)
assert out.Y.shape == (1, 3, 3, 8)
print(f'Vertex states {tuple(out.X.shape)}')
print(f'Ordered-pair states {tuple(out.Y.shape)}')